<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/I-Powered%20Video%20Surveillance%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Video Surveillance System
This notebook implements a surveillance system using a **CNN-LSTM** hybrid architecture to detect suspicious activities, intrusions, and violence in video streams.

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, applications

print("TensorFlow version:", tf.__version__)
print("OpenCV version:", cv2.__version__)

TensorFlow version: 2.19.0
OpenCV version: 4.13.0


### 1. Define Model Constants
We need to define the dimensions for frame resizing and the sequence length (number of frames per video clip) for the LSTM.

In [2]:
IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64
SEQUENCE_LENGTH = 20
CLASSES_LIST = ["Suspicious", "Intrusion", "Violence", "Normal"]

### 2. CNN + LSTM Model Architecture
We will use a `TimeDistributed` wrapper around a CNN to extract features from each frame, then pass those features to an LSTM layer to understand the temporal movement.

In [3]:
def create_model():
    model = models.Sequential()

    # CNN Feature Extraction
    model.add(layers.TimeDistributed(layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
                                     input_shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)))
    model.add(layers.TimeDistributed(layers.MaxPooling2D((4, 4))))
    model.add(layers.TimeDistributed(layers.Dropout(0.25)))

    model.add(layers.TimeDistributed(layers.Flatten()))

    # LSTM Temporal Learning
    model.add(layers.LSTM(32))

    # Output Layer
    model.add(layers.Dense(len(CLASSES_LIST), activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=["accuracy"])
    return model

surveillance_model = create_model()
surveillance_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 64, 64, 16) │           448 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 16, 16, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 16, 16, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 20, 4096)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │       528,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,092 (2.02 MB)

 Trainable params: 529,092 (2.02 MB)

 Non-trainable params: 0 (0.00 B)

### 3. Video Preprocessing Function
This function will read a video, extract frames at regular intervals, and prepare them for model prediction.

In [6]:
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count/SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list

### 4. Inference Logic
Next, we'll set up a function to perform inference on a video file and display the predicted category.

In [7]:
def predict_on_video(video_file_path):
    frames = frames_extraction(video_file_path)
    if len(frames) == SEQUENCE_LENGTH:
        # Reshape to (1, SEQUENCE_LENGTH, HEIGHT, WIDTH, CHANNELS)
        predicted_labels_probabilities = surveillance_model.predict(np.expand_dims(frames, axis=0))[0]
        predicted_label = np.argmax(predicted_labels_probabilities)
        predicted_class_name = CLASSES_LIST[predicted_label]
        print(f'Action Predicted: {predicted_class_name}')
        print(f'Confidence: {predicted_labels_probabilities[predicted_label]}')
    else:
        print("Error: Could not extract enough frames from the video.")

### 3. Video Preprocessing Function
This function will read a video, extract frames at regular intervals, and prepare them for model prediction.

In [4]:
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count/SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list

### 4. Inference Logic
Next, we'll set up a function to perform inference on a video file and display the predicted category.

In [5]:
def predict_on_video(video_file_path):
    frames = frames_extraction(video_file_path)
    if len(frames) == SEQUENCE_LENGTH:
        # Reshape to (1, SEQUENCE_LENGTH, HEIGHT, WIDTH, CHANNELS)
        predicted_labels_probabilities = surveillance_model.predict(np.expand_dims(frames, axis=0))[0]
        predicted_label = np.argmax(predicted_labels_probabilities)
        predicted_class_name = CLASSES_LIST[predicted_label]
        print(f'Action Predicted: {predicted_class_name}')
        print(f'Confidence: {predicted_labels_probabilities[predicted_label]}')
    else:
        print("Error: Could not extract enough frames from the video.")

### 5. Testing the Model
We will download a sample video or you can upload your own to test the detection logic.

In [ ]:
# Download a sample video for testing
sample_video_url = 'https://www.sample-videos.com/video321/mp4/720/big_buck_bunny_720p_1mb.mp4'
video_file_path = 'test_video.mp4'

print("Downloading sample video...")
os.system(f'wget {sample_video_url} -O {video_file_path}')

print("Running inference...")
predict_on_video(video_file_path)

### 5. Testing the Model
We will download a sample video (if available) or you can upload your own to test the detection logic.

In [8]:
# Placeholder: Replace with a valid video URL or local path
sample_video_url = 'https://www.sample-videos.com/video321/mp4/720/big_buck_bunny_720p_1mb.mp4'
video_file_path = 'test_video.mp4'

# Download sample video
os.system(f'wget {sample_video_url} -O {video_file_path}')

# Run Prediction
predict_on_video(video_file_path)

Error: Could not extract enough frames from the video.


In [ ]:
# Download a sample video for testing
sample_video_url = 'https://www.sample-videos.com/video321/mp4/720/big_buck_bunny_720p_1mb.mp4'
video_file_path = 'test_video.mp4'

print('Downloading sample video...')
os.system(f'wget {sample_video_url} -O {video_file_path}')

print('Running inference...')
predict_on_video(video_file_path)